# Target-Supervised Contrastive Embedding

Trains the encoder with `ts_embed.loss.SupConLoss`: the **target label** guides
the contrastive objective — samples with the **same** label are pulled
together, **different** labels are pushed apart. The two masked/unmasked views
of a customer are also mutual positives, so augmentation-invariance and
target-discrimination are learned together.

Result: an embedding whose geometry is aligned to the target, suitable as a
feature block for a downstream model (XGBoost, logistic regression, ...).

Pipeline:
1. Synthetic series + a binary target driven by latent risk factors.
2. Encoder + `SupConLoss`, class-balanced batches.
3. Train; monitor a held-out linear-probe AUC.
4. Evaluate target-discriminability of the learned embedding.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEncoder, TSEncoderConfig
from ts_embed.loss import SupConLoss

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Synthetic data + binary target

Same generator as the fine-tuning notebook: three latent factors drive feature
blocks and an imbalanced binary target.

In [ ]:
N, T, F_NUM, F_CAT = 6000, 24, 98, 2
rng = np.random.default_rng(0)

f_delinq  = rng.standard_normal(N).astype(np.float32)
f_payment = rng.standard_normal(N).astype(np.float32)
f_balance = rng.standard_normal(N).astype(np.float32)

numeric = (0.3 * rng.standard_normal((N, T, F_NUM))).astype(np.float32)
t_axis = np.linspace(0, 1, T, dtype=np.float32)[None, :, None]
numeric[:, :, 0:33]  += f_balance[:, None, None] * (0.5 + t_axis)
numeric[:, :, 33:66] += f_payment[:, None, None] + 0.3 * rng.standard_normal(
    (N, T, 33)).astype(np.float32)
numeric[:, :, 66:98] += f_delinq[:, None, None] * t_axis * 1.5

missing = (rng.random((N, T, F_NUM)) < 0.1).astype(np.uint8)
feat_mean = numeric.mean((0, 1), keepdims=True)
numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)
categorical = np.stack([
    (rng.random((N, T)) < 0.5).astype(np.int8),
    (rng.random((N, T)) < 0.5).astype(np.int8),
], axis=-1)

logit = 1.3 * f_delinq - 1.0 * f_payment + 0.6 * f_balance - 1.4
prob = 1.0 / (1.0 + np.exp(-logit))
target = (rng.random(N) < prob).astype(np.int64)

data_dir = REPO_ROOT / 'data_demo'; data_dir.mkdir(exist_ok=True)
np.save(data_dir / 'numeric.npy', numeric)
np.save(data_dir / 'missing.npy', missing)
np.save(data_dir / 'categorical.npy', categorical)
np.save(data_dir / 'target.npy', target)
print(f'N={N}  target positive rate = {target.mean():.3f}')

## 2. Encoder + SupConLoss + class-balanced loader

`SupConLoss` carries its own projection head; the deliverable embedding is the
**encoder output**, not the projection. We assume you already have target
labels per row for the **training** and **holdout** splits
(`target_train`, `target_holdout`), so each item in the dataset carries its
own target -- no global-index lookup or custom collator class is needed. With
an imbalanced target we use a `WeightedRandomSampler` so each batch has enough
same-(minority)-label pairs for the supervised positives to be meaningful.

In [ ]:
paths = DatasetPaths(numeric=data_dir / 'numeric.npy',
                     missing=data_dir / 'missing.npy',
                     categorical=data_dir / 'categorical.npy')
base_ds = TimeSeriesDataset(paths)

# >>> In production: supply `target_train` and `target_holdout` directly,
#     together with the row indices each refers to. For this synthetic demo we
#     load the saved target array and slice it.
all_targets = torch.from_numpy(np.load(data_dir / 'target.npy')).long()
perm = np.random.default_rng(0).permutation(N)
train_idx   = perm[:int(0.85 * N)]
holdout_idx = perm[int(0.85 * N):]
target_train   = all_targets[torch.from_numpy(train_idx)]
target_holdout = all_targets[torch.from_numpy(holdout_idx)]
# <<<


class TargetDataset(Dataset):
    """Wraps a base dataset so each item yields its target alongside the
    feature dict. With this in place, no index bookkeeping is needed: the
    target travels with the row."""
    def __init__(self, base, targets):
        self.base = base
        self.targets = targets

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        item = self.base[i]
        item['target'] = self.targets[i]
        return item


train_ds = TargetDataset(Subset(base_ds, train_idx.tolist()), target_train)

masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                           n_time_spans=2, feat_span_frac=0.5)
_cc = ContrastiveCollator(masker)

def collate(batch):
    """Build the two contrastive views and stack the per-row targets."""
    targets = torch.stack([b.pop('target') for b in batch])
    out = _cc(batch)
    out['target'] = targets
    return out

# Inverse-frequency weights over the train split for class-balanced batches.
cls_count = torch.bincount(target_train, minlength=2).float()
sample_w = (1.0 / cls_count)[target_train]
assert len(sample_w) == len(train_ds)
sampler = WeightedRandomSampler(sample_w, num_samples=len(train_ds), replacement=True)

loader = DataLoader(train_ds, batch_size=256, sampler=sampler, drop_last=True,
                    num_workers=0, collate_fn=collate)

cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                      d_model=128, n_layers=3, n_heads=4, proj_dim=128)
encoder = TSEncoder(cfg).to(device)
supcon = SupConLoss(in_dim=cfg.d_model, proj_dim=128, proj_hidden=256,
                    temperature=0.1).to(device)
params = list(encoder.parameters()) + list(supcon.parameters())
optim = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
print('batches/epoch:', len(loader),
      '| train pos rate:', round(float(target_train.float().mean()), 3))

## 3. Training loop

Each step encodes both views and applies `SupConLoss` with the target labels.
Every few epochs we fit a logistic regression on the current embedding
(train split) and report its AUC on the held-out split — this tracks whether
the embedding is becoming target-discriminative.

In [ ]:
def roc_auc(y, s):
    y = np.asarray(y); s = np.asarray(s)
    try:
        from sklearn.metrics import roc_auc_score
        return float(roc_auc_score(y, s))
    except ImportError:
        order = np.argsort(s, kind='mergesort')
        ranks = np.empty(len(s), float)
        sr = s[order]; i = 0
        while i < len(sr):
            j = i
            while j + 1 < len(sr) and sr[j + 1] == sr[i]:
                j += 1
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0
            i = j + 1
        pos = y == 1
        npos = int(pos.sum()); nneg = len(y) - npos
        if npos == 0 or nneg == 0:
            return float('nan')
        return float((ranks[pos].sum() - npos * (npos + 1) / 2) / (npos * nneg))

@torch.no_grad()
def all_embeddings():
    """Encoder embedding for every account (unmasked view)."""
    encoder.eval()
    num = np.load(data_dir / 'numeric.npy', mmap_mode='r')
    mis = np.load(data_dir / 'missing.npy', mmap_mode='r')
    cat = np.load(data_dir / 'categorical.npy', mmap_mode='r')
    Z = []
    for i in range(0, N, 512):
        sl = slice(i, min(i + 512, N))
        n = torch.from_numpy(np.asarray(num[sl], dtype=np.float32)).to(device)
        m = torch.from_numpy(np.asarray(mis[sl], dtype=np.float32)).to(device)
        c = torch.from_numpy(np.asarray(cat[sl], dtype=np.int64)).to(device)
        Z.append(encoder(n, m, c).float().cpu().numpy())
    return np.concatenate(Z)

def probe_auc():
    Z = all_embeddings()
    try:
        from sklearn.linear_model import LogisticRegression
        lp = LogisticRegression(max_iter=2000, class_weight='balanced')
        lp.fit(Z[train_idx], target_train.numpy())
        return roc_auc(target_holdout.numpy(),
                       lp.predict_proba(Z[holdout_idx])[:, 1])
    except ImportError:
        return float('nan')

EPOCHS = 15
for epoch in range(EPOCHS):
    encoder.train(); supcon.train()
    last = 0.0
    for batch in loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)
        tgt = batch['target'].to(device)

        emb_a = encoder(num_a, mis_a, cat_a, keep_a)
        emb_b = encoder(num_b, mis_b, cat_b, keep_b)
        loss = supcon(emb_a, emb_b, tgt)['loss']

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optim.step()
        last = float(loss)
    msg = f'epoch {epoch:2d}  supcon_loss={last:.3f}'
    if epoch % 3 == 0 or epoch == EPOCHS - 1:
        msg += f'  held-out probe AUC={probe_auc():.4f}'
    print(msg)

## 4. Is the embedding target-aligned?

Three checks on the held-out split:
1. **Linear-probe AUC** — logistic regression on the embedding.
2. **kNN AUC** — cosine-kNN vote; SupCon optimizes exactly this geometry.
3. **Collapse guard** — per-dimension std must stay > 0.

In [ ]:
Z = all_embeddings()
ytr = target_train.numpy()
yte = target_holdout.numpy()
Ztr, Zte = Z[train_idx], Z[holdout_idx]

# 1. linear probe
try:
    from sklearn.linear_model import LogisticRegression
    lp = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Ztr, ytr)
    lin_auc = roc_auc(yte, lp.predict_proba(Zte)[:, 1])
except ImportError:
    lin_auc = float('nan')

# 2. cosine kNN vote
def knn_score(Xtr, ytr, Xte, k=25):
    Xtr = Xtr / (np.linalg.norm(Xtr, axis=1, keepdims=True) + 1e-8)
    Xte = Xte / (np.linalg.norm(Xte, axis=1, keepdims=True) + 1e-8)
    sims = Xte @ Xtr.T
    nn = np.argpartition(-sims, k, axis=1)[:, :k]
    return ytr[nn].mean(axis=1)  # fraction of positive neighbours

knn_auc = roc_auc(yte, knn_score(Ztr, ytr, Zte))

print(f'held-out linear-probe AUC : {lin_auc:.4f}')
print(f'held-out kNN AUC          : {knn_auc:.4f}')

std = Z.std(0)
print(f'embedding per-dim std: mean={std.mean():.3f} min={std.min():.3f}')
assert std.mean() > 1e-2, 'COLLAPSE: embedding nearly constant'

# Class-mean separation on the train split.
mu_pos = Ztr[ytr == 1].mean(0); mu_neg = Ztr[ytr == 0].mean(0)
between = np.linalg.norm(mu_pos - mu_neg)
within = np.linalg.norm(Ztr - np.where(ytr[:, None] == 1, mu_pos, mu_neg),
                       axis=1).mean()
print(f'class-mean separation ratio = {between / within:.2f} '
      f'(higher = more target-aligned)')

## Next steps

- **Downstream use**: this embedding is target-aligned — feed it to your
  XGBoost (ideally *concatenated* with existing aggregated features, not
  replacing them) and check grouped feature importance / SHAP.
- **Imbalance**: the `WeightedRandomSampler` here balances batches; for very
  rare targets also consider a larger batch size so each batch holds enough
  minority positives.
- **Temperature**: lower `temperature` (0.05-0.1) sharpens the contrast; tune
  on the held-out probe AUC.
- **Combine objectives**: SupCon can be added to the self-supervised VICReg /
  structured losses (weighted sum) so the embedding keeps general structure
  while gaining target alignment.
- **Scale-out**: for DDP, gather projections across ranks (mirror VICReg's
  `_maybe_all_gather`) so SupCon sees more negatives per step.